In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("MASSIVE_API_KEY")

In [ ]:
bool(api_key) # verify credentials exist

In [ ]:
import requests

In [ ]:
# massive.com url using f-string

## wrong: massiveURL = https://api.massive.com/v3/reference/tickers?apiKey=api_key

# right: https://api.massive.com/v2/aggs/ticker/{stocksTicker}/range/{multiplier}/{timespan}/{from}/{to} 
# everything in braces in the above url is a dynamic python expression, every other string is hardcoded!!!

tickerList = ["AAPL", "SNDK", "ISRG", "SHW", "ULTA"]
multiplier = 1
timespan = "day"
startDate = "2026-11-02"
endDate = "2026-11-06"

massiveURL = f"https://api.massive.com/v2/aggs/ticker/{tickerList[0]}/range/{multiplier}/{timespan}/{startDate}/{endDate}" 

print(massiveURL)

In [ ]:
# kwargs = keyword arguments

headers = {"Authorization" : f"Bearer {api_key}"}
params = {"adjusted" : "true", "sort" : "asc"} # adjusted for stock splits and returned in ascending order

In [ ]:
# request

r = requests.get(massiveURL, params, headers=headers)
r.status_code

In [ ]:
def get_request(url, params, **kwargs):
    requestObject = requests.get(url, params, **kwargs)
    return requestObject

In [ ]:
r = get_request(massiveURL, params, headers=headers)

In [ ]:
type(r)

In [ ]:
r.json()

In [ ]:
# practice grabbing closing prices, deserializing, and printing relevant information

closingPrices=[]

stockData = r.json() # converting from json to python obj aka deserialization, from python to json is called serialization

print(f"Number of keys in the dictionary: {len(stockData)}") # it has 8 keys, but I'm interested in the results key
print()

stockData["results"][0] #the first day of the results array which includes OHLC

num_days_returned = stockData["resultsCount"]

keyOfInterest = "results"

for day in stockData[keyOfInterest]:
    closingPrices.append(day["c"])

print(f"Closing prices for {tickerList[0]} from {startDate} to {endDate}: {closingPrices}")
print()
print(f"After converting the response object to a usable python dictionary, these are the keys {stockData.keys()}")
print()
print(f"Number of days requested: {num_days_returned} days")

In [ ]:
# understanding timestamps 
# unix timestamp is elapsed time since 0000 UTC, 01/01/1970

day_1 = stockData["results"][0]
day_1_time_ms = stockData["results"][0]["t"] # unix timestamp in milliseconds

In [ ]:
#timestamp conversion from zulu time to 2026-11-02, for example

from datetime import datetime, timezone
from zoneinfo import ZoneInfo

def extract_trading_dates(stockDictionary):
    dates = []
    for day in stockDictionary["results"]:
        milliseconds = day["t"]
        seconds = milliseconds/1000
        dt_ny = datetime.fromtimestamp(seconds, tz=ZoneInfo("America/New_York"))
        dates.append(dt_ny.date())
    return dates
        
aapl_dates = extract_trading_dates(stockData)
print(aapl_dates)